# 9. Sea ice: onto the 2.5° grid, against ERA5, and with the temperature changes

The sea ice comes into this project three ways:

1. **Where the ice edge is**, on the temperature figures: notebook 04 draws it on its maps (and section 7's zonal means, after Bracegirdle et al., 2024), and notebook 07's D panels use the change in concentration.
2. **Is the models' sea ice realistic?** Each model's historical sea ice against ERA5's, as notebook 02 does for temperature.
3. **How it goes with the temperature changes**: where the ice is lost, does the mean warm more, and does the distribution narrow (the cold tail warming most)?

| Section | What | Saved, under `results/siconc/<RUN>/` |
|---|---|---|
| 1. Onto the 2.5° grid | how one model's field is put on the temperature grid, step by step | |
| 2. The data | the sample, or the CEDA files and ERA5 (JASMIN), on the 2.5° grid | full: monthly stores, like the temperature's |
| 3. Extent and the ice edge | each member's seasonal extent, and the concentration over the final years | `extent`, `climatology` (read by notebooks 04 and 07) |
| 4. Against ERA5 | the seasonal cycle, where the edge is, and the extent's mean, variability and trend | `era5_extent`, `era5_climatology`, `historical_climatology`, `era5_extent_test` |
| 5. With the temperature changes | the change in concentration, the mean and width changes in each zone, across the models, and from year to year | `ice_change`, `zone_means`, `ice_tas_correlation` |

**Where the sea ice comes from.** Not the LEADER-EPESC folders: CEDA's copy of the CMIP6 archive (`paths.CEDA_CMIP6`), DAMIP for the single forcings and CMIP for historical, each model on its own ocean grid (`siconc`). GISS-E2-1-G has only its atmosphere grid (`siconca`), which is 0 rather than empty over land, so it takes ERA5's land. CEDA has far fewer members than the temperature files (section 2 has the table), MPI-ESM1-2-LR has only historical, and GISS-E2-1-G has no hist-nat. CanESM5 and MIROC6 have **hist-stratO3** (stratospheric ozone only) rather than hist-totalO3: here it stands in for their ozone run (`OZONE_NAMES`). Check that against what LEADER-EPESC's hist-totalO3 is for these two models.

**Everything is on the 2.5° grid** of the temperature, so the edge can be drawn on the temperature maps and the two compared cell by cell. Each cell also keeps how much of it is ocean, so that the extent counts the ocean of a coastal cell and not its land (section 1).

**The sample** (`tests/data/sea_ice`, made by `scripts/sea_ice_sample.py`) is one member of each model and experiment for 2013–2014, and ERA5 at 00 UTC on the 15th of each month. So on the sample "the final years" are one year, the sections that need years or members are JASMIN only, and the temperature sample (a 3×3 block of cells on the plateau, away from any ice) leaves section 5's figures nearly empty.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from extant import config, convert, loading, paths, preprocessing as pp, sea_ice, significance as sig, storage
from extant import era5_evaluation as ev
from extant.datatree import reduce_to_dataset, skip_empty, tree_to_dataset
from extant.logger import setup_logging
from extant.plots import era5_evaluation as pe, forced_response as fr, paper_panels, sea_ice as sip

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)
pd.set_option('display.max_rows', 200)

In [ ]:
%matplotlib inline

## Settings

In [ ]:
#(t): The sea ice, and the temperature whose changes section 5 compares it with
SEA_ICE = 'siconc'
VARIABLE = 'tas'
TAS_UNITS = config.VARIABLES[VARIABLE].units

#(t): The models analysed (notebook 01), the experiments in the order of every figure, and the seasons
MODELS = ['CanESM5', 'GISS-E2-1-G', 'HadGEM3-GC31-LL', 'MIROC6', 'MPI-ESM1-2-LR', 'NorESM2-LM']
EXPERIMENTS = ['hist-GHG', 'hist-aer', 'hist-totalO3', 'historical']
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The model and experiment of the one-model figures
MODEL = config.PLOT_SEL['model']
EXPERIMENT = 'hist-GHG'

#(t): CanESM5 and MIROC6 have hist-stratO3 on CEDA rather than hist-totalO3: it stands in for their ozone run
OZONE_NAMES = {'hist-stratO3': 'hist-totalO3'}

#(t): Concentration (%) from which a cell counts as ice
THRESHOLD = sea_ice.THRESHOLD

#(t): The final years compared with hist-nat (as in notebook 03), and the years compared with ERA5: the satellite era
#(t): up to the last whole year of historical (DJF 2013)
WINDOW = config.WINDOW
ERA5_YEARS = slice(config.ERA5_START_YEAR, 2013)

#(t): The ERA5 test's two-sided significance level: ERA5 outside the members' 5-95%, as in notebook 02
ERA5_ALPHA = ev.ALPHA


def result_path(name):
    #(t): Where this notebook saves a result: results/siconc/<RUN>/<name>.zarr
    return paths.result(name, SEA_ICE, RUN)


def tas_result_path(name):
    #(t): Where notebook 03 saved a temperature result: results/tas/<RUN>/<name>.zarr
    return paths.result(name, VARIABLE, RUN)

### Colour scales

One per quantity. A difference in concentration is blue where there is more ice and red where there is less, so that ice lost reads like warming on the temperature maps.

In [ ]:
concentration_scale = fr.Scale(np.arange(0, 101, 10), 'Blues', 'Sea-ice concentration (%)', extend='neither')
concentration_change_scale = fr.Scale(np.arange(-55, 56, 10), 'RdBu', 'Difference in sea-ice concentration (%)',
                                      ticks=np.arange(-50, 51, 10))
correlation_scale = fr.Scale(np.round(np.arange(-1, 1.01, 0.2), 1), 'PuOr_r', 'Correlation of sea ice and temperature',
                             extend='neither')
zone_scale = fr.class_scale(sea_ice.ZONES, ['#f4f3ef', '#2c7fb8', '#e6550d', '#31a354', '#bdbdbd'], 'Zone')
tas_scales = fr.colour_scales(VARIABLE)

## Dask cluster (JASMIN)

For section 2's full data: the CEDA files and the hourly ERA5 files.

In [ ]:
from extant import jasmin

gateway, cluster, client = jasmin.connect(n_workers=20, wait_for=6)
jasmin.upload_package(client)
client

## 1. How a field goes onto the 2.5° grid

Each model's sea ice is on its own ocean grid: about 1°, curvilinear (each point has its own latitude and longitude), with its own cell areas (`areacello`). The temperature is on a regular 2.5° grid. A 2.5° cell $C$ gets the area-weighted mean of the points $k$ whose centres fall in it:

$$
c_C = \frac{\sum_{k \in C} a_k\, c_k}{\sum_{k \in C} a_k},
$$

leaving out points with no value (land). Where the points are much smaller than the cells (an ocean model's 1°, ERA5's 0.25°) this is the conservative regridding notebook 01 does for ERA5's temperature, give or take the points that straddle a cell edge.

**Coastal cells.** A cell on the coast is part land, and its concentration is the mean over its ocean only. Counted whole, it would add its land to the extent: about 3 million km² right round the continent (ERA5 in September 2013 gives 22.2 million km² instead of the 19.3 of its own grid). So each cell also keeps its **ocean area**, the summed area of its ocean points,

$$
A_C = \sum_{k \in C,\ \mathrm{ocean}} a_k ,
$$

and the extent counts that, $\mathrm{SIE} = \sum_C A_C\,[c_C \ge 15\%]$, with the rest of each cell south of 60°S the continent. ERA5 in September 2013 then gives 19.9 million km²: the rest of the difference is the cells the 15% line runs through, which count all their ocean.

Step by step, on one month of `MODEL`'s historical member in the sample (the sample is in this repository, so this runs anywhere).

In [ ]:
#(t): MODEL's historical file in the sample, and the file of its grid's cell areas
example_files_df = loading.sea_ice_sample_files().query('model == @MODEL and experiment == "historical"')
example_files_df

In [ ]:
example_path = example_files_df['path'].iloc[0]
example_area_path = example_files_df['area_path'].iloc[0]
example_path, example_area_path

In [ ]:
#(t): The concentration on the model's own grid: (time, j, i), each point with its own latitude and longitude
example_native_ds = xr.open_dataset(example_path)
example_native_ds

In [ ]:
example_area_ds = xr.open_dataset(example_area_path)
example_area_ds

In [ ]:
#(t): Where the points are, from the cell-area file (NorESM2-LM's sea-ice files leave the positions empty over land)
example_lat_da, example_lon_da = sea_ice.native_lat_lon(example_area_ds)
example_lat_da

In [ ]:
#(t): Each point's area, in km² (CMIP's are in m²)
example_area_km2_da = example_area_ds['areacello'] / 1e6
example_area_km2_da

In [ ]:
#(t): Quick look: the points' latitudes and areas on the model's own (j, i) grid
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
example_lat_da.plot(ax=axes[0])
example_area_km2_da.plot(ax=axes[1]);

In [ ]:
#(t): Which 2.5° cell each point falls in, numbered row by row (-1: outside the grid)
example_boxes_da = sea_ice.box_index(example_lat_da, example_lon_da)
example_boxes_da

In [ ]:
example_boxes_da.plot();

In [ ]:
#(t): One month, September 2013, on the model's own grid
example_native_da = example_native_ds['siconc'].sel(time='2013-09').squeeze('time', drop=True)
example_native_da

In [ ]:
#(t): ... and on the 2.5° grid: each cell the area-weighted mean of its points
example_grid_da = sea_ice.to_grid(example_native_da, example_boxes_da, example_area_km2_da)
example_grid_da

In [ ]:
#(t): Quick look: before and after
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
example_native_da.plot(ax=axes[0], vmin=0, vmax=100)
example_grid_da.plot(ax=axes[1], vmin=0, vmax=100);

In [ ]:
#(t): How much of each 2.5° cell is ocean (km²): the summed areas of the points with a concentration
example_ocean_area_da = sea_ice.ocean_area_on_grid(example_native_da.notnull(), example_boxes_da, example_area_km2_da)
example_ocean_area_da

In [ ]:
#(t): As a share of each cell: 1 in the open ocean, less on the coast, 0 over the continent. A share a little over 1
#(t): is a cell that got one more row of points than its neighbour; the areas still add up to the model's ocean
example_ocean_share_da = example_ocean_area_da / sea_ice.cell_area(example_ocean_area_da['lat'], example_ocean_area_da['lon'])
example_ocean_share_da.plot(vmin=0, vmax=1.3);

In [ ]:
fig, axes = sip.regrid_schematic(example_native_da, example_lat_da, example_lon_da, example_grid_da,
                                 title=f'How a field goes onto the 2.5° grid: {MODEL}, historical, September 2013')

In [ ]:
#(t): The extent (million km²) three ways: on the model's own grid; on the 2.5° grid counting whole cells; and
#(t): counting only each cell's ocean, as everything below does
example_extents_df = pd.Series({
    'own grid': float((example_area_km2_da * (example_native_da >= THRESHOLD)).sum()) / 1e6,
    '2.5°, whole cells': float(sea_ice.extent(example_grid_da, sea_ice.cell_area(example_grid_da['lat'], example_grid_da['lon']))) / 1e6,
    '2.5°, each cell\'s ocean': float(sea_ice.extent(example_grid_da, example_ocean_area_da)) / 1e6,
}).to_frame('extent (million km²)').round(2)
example_extents_df

## 2. The data on the 2.5° grid

Run **one** of the next two sections. Both leave

- `ice_monthly_tree`: /<model>/<experiment>, `siconc` (%) on (member, time, lat, lon) and `ocean_area` (km²) on (lat, lon);
- `era5_ice_ds`: ERA5's `siconc` (%) on (time, lat, lon) and its `ocean_area`;
- `RUN`, which says where the results are saved and which temperature results section 5 opens.

### Sample data

In [ ]:
#(t): ERA5's sea-ice cover at 00 UTC on the 15th of each month of 2013-2014, on its 0.25° grid, as a fraction (0-1)
era5_sample_path = paths.SEA_ICE_SAMPLE_DIR / 'era5_ci.nc'
era5_sample_path

In [ ]:
era5_native_ds = xr.open_dataset(era5_sample_path).rename(latitude='lat', longitude='lon')
era5_native_ds

In [ ]:
#(t): Which 2.5° cell each ERA5 point falls in
era5_boxes_da = sea_ice.box_index(era5_native_ds['lat'], era5_native_ds['lon'])

In [ ]:
#(t): Each ERA5 point's area (km²)
era5_area_da = sea_ice.cell_area(era5_native_ds['lat'], era5_native_ds['lon'])

In [ ]:
#(t): ERA5 on the 2.5° grid, still a fraction
era5_fraction_da = sea_ice.to_grid(era5_native_ds['siconc'], era5_boxes_da, era5_area_da)

In [ ]:
#(t): In %, like the models
era5_percent_da = 100 * era5_fraction_da

In [ ]:
#(t): Each cell's ocean (km²): the ERA5 points with a value, as ERA5 leaves land empty
era5_ocean_area_da = sea_ice.ocean_area_on_grid(era5_native_ds['siconc'].isel(time=0).notnull(), era5_boxes_da,
                                                era5_area_da)

In [ ]:
era5_ice_ds = xr.Dataset({'siconc': era5_percent_da, 'ocean_area': era5_ocean_area_da})
era5_ice_ds

In [ ]:
#(t): The models' files: one member of each model and experiment, on its own grid, with its cell-area file
sample_files_df = loading.sea_ice_sample_files()
sample_files_df

In [ ]:
#(t): hist-stratO3 stands in for hist-totalO3 (OZONE_NAMES)
sample_files_df['experiment'] = sample_files_df['experiment'].replace(OZONE_NAMES)

In [ ]:
#(t): One file per model and experiment: siconc (the ocean grid) where a model has both it and siconca
sample_files_df = sample_files_df.sort_values(['model', 'experiment', 'variable']).drop_duplicates(['model', 'experiment'])
sample_files_df

In [ ]:
%%time
#(t): Every file onto the 2.5° grid. GISS-E2-1-G's siconca is 0 over land rather than empty, so it takes ERA5's ocean
ice_monthly_tree = xr.DataTree.from_dict({
    f'{row.model}/{row.experiment}': sea_ice.open_on_grid(
        {row.member: [row.path]}, row.area_path, row.variable,
        ocean_area=era5_ocean_area_da if row.variable == 'siconca' else None)
    for row in sample_files_df.itertuples()
}).compute()
ice_monthly_tree

In [ ]:
RUN = 'sample'

### Full data (JASMIN)

The CEDA files of every member onto the 2.5° grid, saved as monthly stores alongside the temperature's (`paths.lesfmip_monthly(model, 'siconc', experiment, config.GROUP)`), once; and ERA5's hourly files to monthly means on the grid. Both take a while and skip what is already done.

#### What is on CEDA

In [ ]:
#(t): Every folder of the sea ice and the cell areas of the models and experiments (the areas also from piControl), in
#(t): DAMIP, CMIP and CMIP6Plus
CEDA_EXPERIMENTS = [*config.MAIN_EXPERIMENTS, *OZONE_NAMES]
ice_inventory_df = convert.ceda_inventory(MODELS, CEDA_EXPERIMENTS)
ice_inventory_df

In [ ]:
#(t): Members of each model (rows) and variable and experiment (columns)
ice_inventory_df.query('table == "SImon"').pivot_table(index=['project', 'model'], columns=['variable', 'experiment'],
                                                       values='member', aggfunc='nunique')

In [ ]:
#(t): The members used: CMIP6 only (CMIP6Plus's f2023 members have other forcing and are not pooled with them)
ice_members_df = ice_inventory_df.query('table == "SImon" and project == "CMIP6"')

In [ ]:
#(t): siconc where a model has it for an experiment, else siconca ('siconc' sorts first, so it is the minimum)
ice_members_df = ice_members_df[ice_members_df['variable']
                                == ice_members_df.groupby(['model', 'experiment'])['variable'].transform('min')]

In [ ]:
#(t): Only members that cover 1850 to the end of historical
ice_members_df = ice_members_df.query('start <= 1850 and end >= 2014')
ice_members_df

In [ ]:
#(t): The left-out members, if any: a record that starts late or stops early
ice_inventory_df.query('table == "SImon" and project == "CMIP6" and (start > 1850 or end < 2014)')

#### ERA5: daily values to monthly means (JASMIN)

ERA5's sea ice is taken from satellite analyses once a day, so the 00 UTC field is the day's: each month's mean is of about 30 files rather than 720. Each month is averaged on a worker, put on the 2.5° grid in %, and each year saved as a store, then joined, as in notebook 01, section 2. ERA5 comes first: GISS-E2-1-G takes its land.

In [ ]:
#(t): One day's file, for the grid and the land
era5_hour_path = convert.era5_files(SEA_ICE, config.ERA5_START_YEAR, 1, hour=0)[0]
era5_hour_path

In [ ]:
era5_hour_ds = convert.era5_south(xr.open_dataset(era5_hour_path)).rename(latitude='lat', longitude='lon')
era5_hour_ds

In [ ]:
era5_boxes_da = sea_ice.box_index(era5_hour_ds['lat'], era5_hour_ds['lon'])

In [ ]:
era5_area_da = sea_ice.cell_area(era5_hour_ds['lat'], era5_hour_ds['lon'])

In [ ]:
#(t): Each cell's ocean (km²): the points with a value
era5_ocean_area_da = sea_ice.ocean_area_on_grid(era5_hour_ds[config.VARIABLES[SEA_ICE].era5_name].isel(time=0).notnull(),
                                                era5_boxes_da, era5_area_da)
era5_ocean_area_da

In [ ]:
#(t): Years already done: saved as a year store (an interrupted run), or in the joined store
era5_years_dir = paths.era5_years_dir(SEA_ICE)
era5_done_years = [int(store.stem) for store in paths.find_all(era5_years_dir, '*.zarr')]
if storage.exists(paths.era5_monthly(SEA_ICE)):
    era5_done_years += np.unique(storage.open_dataarray(paths.era5_monthly(SEA_ICE)).time.dt.year).tolist()
era5_done_years

In [ ]:
#(t): The years left, up to the end of historical
era5_todo_years = [year for year in range(config.ERA5_START_YEAR, 2015) if year not in era5_done_years]
era5_todo_years

In [ ]:
%%time
#(t): Every month on the workers; each year onto the grid, in %, and saved as soon as its twelve months are back
era5_futures = {year: [client.submit(convert.era5_month_mean, convert.era5_files(SEA_ICE, year, month, hour=0),
                                     config.VARIABLES[SEA_ICE].era5_name, pure=False) for month in range(1, 13)]
                for year in era5_todo_years}
for year in era5_todo_years:
    year_da = xr.DataArray(np.stack(client.gather(era5_futures[year])), dims=('time', 'lat', 'lon'),
                           coords={'time': pd.date_range(f'{year}-01-01', periods=12, freq='MS'),
                                   'lat': era5_hour_ds['lat'], 'lon': era5_hour_ds['lon']})
    year_grid_da = 100 * sea_ice.to_grid(year_da, era5_boxes_da, era5_area_da)
    storage.save(year_grid_da.astype('float32').rename(SEA_ICE), era5_years_dir / f'{year}.zarr')

In [ ]:
#(t): The year stores joined into one store, and deleted
era5_monthly_da = convert.combine_era5_years(SEA_ICE)
era5_monthly_da

#### The models: one store, step by step (JASMIN)

The stores left to make, under the names of notebook 01's (hist-stratO3 saved as hist-totalO3). To make one again, delete it first.

In [ ]:
#(t): One row per model and experiment: its variable and grid, and the store it goes to
ice_stores_df = ice_members_df.drop_duplicates(['model', 'experiment'])[['model', 'experiment', 'variable', 'grid']]
ice_stores_df = ice_stores_df.assign(store=[paths.lesfmip_monthly(model, SEA_ICE, OZONE_NAMES.get(experiment, experiment), config.GROUP)
                                            for model, experiment in zip(ice_stores_df['model'], ice_stores_df['experiment'])])
ice_stores_df

In [ ]:
#(t): Those not made yet
ice_todo_df = ice_stores_df[[not storage.exists(store) for store in ice_stores_df['store']]]
ice_todo_df

In [ ]:
first_store = ice_todo_df.iloc[0]
first_store

In [ ]:
#(t): Its members' files
first_files = convert.ceda_member_files(ice_members_df.query('model == @first_store.model and experiment == @first_store.experiment'))
first_files

In [ ]:
#(t): The cell areas of its grid
first_area_path = convert.ceda_area_file(ice_inventory_df, first_store.model, sea_ice.AREA_VARIABLES[first_store.variable],
                                         first_store.grid)
first_area_path

In [ ]:
%%time
#(t): All its members on the 2.5° grid (lazily)
first_ds = sea_ice.open_on_grid(first_files, first_area_path, first_store.variable,
                                ocean_area=era5_ocean_area_da if first_store.variable == 'siconca' else None)
first_ds

In [ ]:
first_ds['siconc'].isel(member=0).sel(time='2000-09').plot();

#### The models: all of them (JASMIN)

Every store left, one after the other. If one fails, the loop stops with its error; save that one by hand without its broken members (as notebook 01, section 1 says) and run the loop again.

In [ ]:
%%time
for store in ice_todo_df.itertuples():
    rows_df = ice_members_df.query('model == @store.model and experiment == @store.experiment')
    area_path = convert.ceda_area_file(ice_inventory_df, store.model, sea_ice.AREA_VARIABLES[store.variable], store.grid)
    members_ds = sea_ice.open_on_grid(convert.ceda_member_files(rows_df), area_path, store.variable,
                                      ocean_area=era5_ocean_area_da if store.variable == 'siconca' else None)
    storage.save(members_ds, store.store, consolidated=True)

#### Open them (JASMIN)

In [ ]:
#(t): The monthly sea-ice stores, as a DataTree /<model>/<experiment>
ice_monthly_tree = loading.open_lesfmip_monthly(SEA_ICE, models=MODELS, experiments=config.MAIN_EXPERIMENTS,
                                                groups=[config.GROUP])
ice_monthly_tree

In [ ]:
#(t): ERA5, with each cell's ocean
era5_ice_ds = xr.Dataset({'siconc': loading.open_era5_monthly(SEA_ICE).load(), 'ocean_area': era5_ocean_area_da})
era5_ice_ds

In [ ]:
RUN = 'full'

### What was opened

In [ ]:
#(t): Members of each model (rows) and experiment (columns)
ice_member_counts_df = pd.Series({(node.parent.name, node.name): node.sizes['member']
                                  for node in ice_monthly_tree.leaves}).unstack()
ice_member_counts_df

In [ ]:
#(t): Where each model's sea ice came from: its ocean grid (siconc) or its atmosphere grid (siconca)
pd.Series({node.parent.name: node['siconc'].attrs.get('source_variable') for node in ice_monthly_tree.leaves})

In [ ]:
#(t): Quick look: MODEL's first historical member in September 2013, and ERA5's
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
ice_monthly_tree[MODEL]['historical']['siconc'].isel(member=0).sel(time='2013-09').squeeze('time').plot(
    ax=axes[0], vmin=0, vmax=100)
era5_ice_ds['siconc'].sel(time='2013-09').squeeze('time').plot(ax=axes[1], vmin=0, vmax=100);

## 3. Extent and the ice edge

The **sea-ice extent** is the ocean area of the cells with at least 15% ice, worked out every month and then averaged into seasons, for every member (complete seasons in whole years, cut at `config.LAST_MONTH` like the temperature, so "the final years" are the same years). The **ice edge** on a map is the 15% contour of the ensemble-mean concentration over the final `WINDOW` years. On a zonal mean it is an **equivalent latitude** (Bracegirdle et al., 2024): the latitude whose polar cap holds the extent and the continent together,

$$
\varphi_{\mathrm{eq}} = \arcsin\left(\frac{\mathrm{SIE} + A_{\mathrm{continent}}}{2\pi R^2} - 1\right).
$$

These are what notebook 04 (its maps, and section 7) and notebook 07 (D1–D4) read.

In [ ]:
#(t): Seasonal means of the concentration (%), every member, by the same steps as the temperature's (notebook 01)
ice_season_tree = pp.seasonal_tree(ice_monthly_tree, SEA_ICE)
ice_season_tree

In [ ]:
%%time
#(t): Each member's monthly extent (km²), and each model's continent
ice_monthly_extent_tree = ice_monthly_tree.map_over_datasets(skip_empty(sea_ice.monthly_extent)).compute()
ice_monthly_extent_tree

In [ ]:
%%time
#(t): ... averaged into seasons
ice_extents_tree = sea_ice.seasonal_extent(ice_monthly_tree).compute()
ice_extents_tree

In [ ]:
%%time
#(t): The ensemble-mean concentration over each experiment's final WINDOW years (%): the ice edge of the maps
ice_climatology_da = sea_ice.concentration_climatology(ice_season_tree, years=WINDOW).compute()
ice_climatology_da

In [ ]:
#(t): Quick look: hist-nat's, every model and season
ice_climatology_da.sel(experiment='hist-nat', season=SEASONS).plot(row='model', col='season', vmin=0, vmax=100);

In [ ]:
#(t): The equivalent latitude of each model's ice edge in each experiment's final WINDOW years
ice_edges_da = sea_ice.edge_latitude(ice_extents_tree, years=WINDOW)
ice_edges_da.sel(season=SEASONS).to_dataframe().unstack('season')['edge_latitude'].round(1)

**The method on one field.** `MODEL`'s hist-nat ensemble mean in JJA: the concentration and its 15% contour; the cells that count (ice, and the continent), with the circle that encloses the same area; and the cap area read backwards from their total.

In [ ]:
#(t): MODEL's hist-nat ensemble mean in JJA, and its cells' ocean
example_field_da = ice_climatology_da.sel(model=MODEL, experiment='hist-nat', season='JJA')
example_field_ocean_da = ice_monthly_tree[MODEL]['hist-nat']['ocean_area']

In [ ]:
example_edge_ds = sea_ice.edge_example(example_field_da, ocean_area=example_field_ocean_da)
example_edge_ds

In [ ]:
fig, axes = sip.edge_schematic(example_edge_ds, title=f'How the ice edge becomes a latitude: {MODEL}, hist-nat, '
                                                      f'final {WINDOW} years, JJA')

In [ ]:
#(t): Each experiment's extent through time: the ensemble mean and the 5-95% of members
ice_extent_summary_ds = sea_ice.extent_summary(ice_extents_tree)
ice_extent_summary_ds

In [ ]:
fig, axes = sip.extent_timeseries(ice_extent_summary_ds, seasons=SEASONS,
                                  title='Sea-ice extent, every model, experiment and season')

### Save

In [ ]:
storage.save(ice_extents_tree, result_path('extent'))

In [ ]:
storage.save(ice_climatology_da.rename(SEA_ICE), result_path('climatology'))

## 4. Against ERA5: is the models' sea ice realistic?

Each model's historical members against ERA5 over `ERA5_YEARS` (1979–2013), counted on the same 2.5° grid in the same way. Three questions:

1. **The seasonal cycle**: does the model have the right amount of ice in summer (February) and winter (September)?
2. **Where the edge is**: the 15% contour of the mean concentration, and how far north the edge reaches as an equivalent latitude.
3. **The extent's mean, year-to-year variability and trend**, season by season, tested as notebook 02 tests the temperature (Suarez-Gutierrez et al., 2021): if the model were right, ERA5 would behave like one more member, so its value would fall outside the members' 5–95% only 10% of the time. With $N$ members the smallest two-sided p-value is $2/(N+1)$, so a model needs at least 20 members for anything to be flagged: GISS-E2-1-G (10 on CEDA), MPI-ESM1-2-LR (2) and NorESM2-LM (1) are shown but cannot fail.

The usual CMIP6 picture (Roach et al., 2020, *Geophys. Res. Lett.* 47, e2019GL086729): most models have too little ice in summer and a spread of winter extents round the observed one, and they lose ice over 1979–2014 when the observed extent grew slightly. A model whose edge is far from ERA5's will put its temperature changes at the wrong latitudes too, which is worth knowing before reading section 5 or notebook 04.

On the sample this is one member over one year, so only 4.1 and 4.2 mean anything (and only roughly); 4.3 needs the years and the members (JASMIN).

### ERA5, counted the same way

In [ ]:
#(t): ERA5's monthly extent (km²) and its continent
era5_monthly_extent_ds = sea_ice.monthly_extent(era5_ice_ds)
era5_monthly_extent_ds

In [ ]:
#(t): Its seasonal means, complete seasons in whole years like the models'
era5_extent_ds = pp.seasonal_means(era5_monthly_extent_ds[['extent']]).assign(
    continent_area=era5_monthly_extent_ds['continent_area'])
era5_extent_ds

In [ ]:
#(t): ERA5's seasonal concentration (%), on (year, season, lat, lon)
era5_ice_season_da = pp.seasonal_era5(era5_ice_ds['siconc'], SEA_ICE, like=ice_climatology_da)
era5_ice_season_da

### 4.1 The seasonal cycle

Each historical member's mean extent in each calendar month over `ERA5_YEARS`, against ERA5's.

In [ ]:
#(t): Every historical member's cycle (million km²), on (model, member, month)
historical_cycle_da = reduce_to_dataset(ice_monthly_extent_tree.match('*/historical'),
                                        partial(sea_ice.monthly_cycle, years=ERA5_YEARS))['extent'].sel(experiment='historical')
historical_cycle_da

In [ ]:
#(t): ERA5's
era5_cycle_da = sea_ice.monthly_cycle(era5_monthly_extent_ds, years=ERA5_YEARS)['extent']
era5_cycle_da

In [ ]:
#(t): Quick look: each model's ensemble mean, and ERA5 (black)
historical_cycle_da.mean('member').plot.line(x='month', hue='model')
era5_cycle_da.plot(color='k', lw=3);

In [ ]:
fig, axes = sip.seasonal_cycle(historical_cycle_da, era5_cycle_da,
                               title=f'Seasonal cycle of sea-ice extent, historical against ERA5, '
                                     f'{ERA5_YEARS.start}-{ERA5_YEARS.stop}')

In [ ]:
#(t): The ensemble means in February (the minimum) and September (the maximum), against ERA5's (million km²)
cycle_extremes_df = historical_cycle_da.mean('member').sel(month=[2, 9]).transpose('model', 'month').to_pandas()
cycle_extremes_df.loc['ERA5'] = era5_cycle_da.sel(month=[2, 9]).values
cycle_extremes_df.rename(columns={2: 'February', 9: 'September'}).round(1)

### 4.2 Where the edge is

The historical ensemble-mean concentration over `ERA5_YEARS` minus ERA5's, with both edges: the model's (black) and ERA5's (orange). Blue: the model has more ice than ERA5; red: less.

In [ ]:
#(t): Each model's historical ensemble mean over ERA5_YEARS (%)
historical_climatology_da = sea_ice.concentration_climatology(ice_season_tree.match('*/historical'),
                                                              years=ERA5_YEARS).sel(experiment='historical').compute()
historical_climatology_da

In [ ]:
#(t): ERA5's
era5_climatology_da = era5_ice_season_da.sel(year=ERA5_YEARS).mean('year')
era5_climatology_da

In [ ]:
#(t): The difference
historical_bias_da = historical_climatology_da - era5_climatology_da
historical_bias_da

In [ ]:
#(t): The two edges drawn on each map: the model's, and ERA5's (the same on every row)
model_era5_edges_da = xr.concat([historical_climatology_da, era5_climatology_da.broadcast_like(historical_climatology_da)],
                                dim=pd.Index(['model', 'ERA5'], name='edge'))
model_era5_edges_da

In [ ]:
#(t): Quick look
historical_bias_da.sel(season=SEASONS).plot(row='model', col='season', vmin=-60, vmax=60, cmap='RdBu');

In [ ]:
panels = fr.map_grid(historical_bias_da.sel(season=SEASONS), concentration_change_scale, row_dim='model',
                     col_dim='season', edges=model_era5_edges_da.sel(season=SEASONS),
                     title=f'Sea-ice concentration, historical minus ERA5, {ERA5_YEARS.start}-{ERA5_YEARS.stop}')

### 4.3 The edge as a latitude

The equivalent latitude of each member's edge over `ERA5_YEARS`, and ERA5's.

In [ ]:
#(t): Each historical member's mean extent over ERA5_YEARS, and its model's continent
historical_member_extent_ds = sea_ice.member_means(ice_extents_tree.match('*/historical'),
                                                   ERA5_YEARS).sel(experiment='historical')
historical_member_extent_ds

In [ ]:
#(t): ... as equivalent latitudes
historical_member_edges_da = sea_ice.equivalent_latitude(historical_member_extent_ds['extent']
                                                         + historical_member_extent_ds['continent_area'])
historical_member_edges_da

In [ ]:
#(t): ERA5's mean extent over the same years, and its edge
era5_mean_extent_ds = era5_extent_ds.sel(year=ERA5_YEARS).mean('year')
era5_edge_da = sea_ice.equivalent_latitude(era5_mean_extent_ds['extent'] + era5_mean_extent_ds['continent_area'])
era5_edge_da

In [ ]:
fig, axes = sip.edge_latitudes(historical_member_edges_da.sel(season=SEASONS), era5_edge_da.sel(season=SEASONS),
                               title=f'Where the ice edge is: historical against ERA5, {ERA5_YEARS.start}-{ERA5_YEARS.stop}')

In [ ]:
#(t): How far each model's edge is from ERA5's (degrees; negative: further south, too little ice)
(historical_member_edges_da.mean('member') - era5_edge_da).sel(season=SEASONS).to_pandas().round(1)

In [ ]:
#(t): ... and its continent against ERA5's (million km²): the grids' coasts differ a little
(historical_member_extent_ds['continent_area'] / 1e6).to_series().round(2).to_frame('continent').assign(
    ERA5=float(era5_mean_extent_ds['continent_area']) / 1e6)

### 4.4 The extent's mean, variability and trend: ERA5 among the members (JASMIN)

For each model and season, three statistics of the seasonal extent over `ERA5_YEARS`: its **mean**, its **year-to-year standard deviation** (of the detrended extent), and its **trend**, each worked out for every member and for ERA5, then ERA5's place among the members (`sea_ice.era5_extent_test`, the test of notebook 02).

In [ ]:
%%time
era5_extent_test_ds = reduce_to_dataset(
    ice_extents_tree.match('*/historical'),
    partial(sea_ice.era5_extent_test, era5=era5_extent_ds['extent'], years=ERA5_YEARS, alpha=ERA5_ALPHA),
).sel(experiment='historical')
era5_extent_test_ds

In [ ]:
#(t): ERA5's percentile among each model's members (50: in the middle; 0 or 100: beyond every member)
era5_extent_test_ds['percentile'].sel(season=SEASONS).to_dataframe().unstack(['statistic', 'season'])['percentile'].round(0)

In [ ]:
fig, axes = sip.era5_strips(era5_extent_test_ds.sel(season=SEASONS), 'mean')

In [ ]:
fig, axes = sip.era5_strips(era5_extent_test_ds.sel(season=SEASONS), 'std')

In [ ]:
fig, axes = sip.era5_strips(era5_extent_test_ds.sel(season=SEASONS), 'trend')

In [ ]:
fig, axes = pe.scorecard(era5_extent_test_ds.sel(season=SEASONS),
                         title=f'Sea-ice extent: where ERA5 falls among each model\'s historical members, '
                               f'{ERA5_YEARS.start}-{ERA5_YEARS.stop}')

In [ ]:
storage.save(era5_extent_test_ds, result_path('era5_extent_test'))

### Save

In [ ]:
storage.save(era5_extent_ds, result_path('era5_extent'))

In [ ]:
storage.save(era5_climatology_da.rename(SEA_ICE), result_path('era5_climatology'))

In [ ]:
storage.save(historical_climatology_da.rename(SEA_ICE), result_path('historical_climatology'))

## 5. Sea ice and the temperature changes

What to expect (Bracegirdle et al., 2024). Over sea ice in winter the surface can cool far below freezing, because the ice insulates the air from the ocean; over open water the ocean holds it near the freezing point of sea water (about −1.8 °C). So where the ice is lost, the mean warms most, and the cold tail warms more than the warm one: the distribution **narrows**. Near the edge, where a season can be over ice or open water, the year-to-year variability is largest. The changes should therefore line up with where the ice changed, most in autumn and winter (MAM, JJA, SON), least in summer.

Four ways of looking:

- **5.1** the change in concentration itself (each experiment's final `WINDOW` years minus hist-nat's), with both edges;
- **5.2** the temperature changes of notebook 03 averaged over the zones where the ice stayed, went, came, or never was;
- **5.3** across models and forcings: a bigger loss of ice against a bigger change in the mean and the width;
- **5.4** year to year in hist-nat: where the sea ice and the temperature move together (JASMIN).

### Open the temperature results

Notebook 03's results for the same `RUN` (on the sample, a 3×3 block of cells on the plateau, so most of what follows is empty there).

In [ ]:
#(t): The mean change (hist-nat bootstrap), the change in width Q95 - Q05, and the change in each tail
mean_bootstrap_ds = storage.open_dataset(tas_result_path('mean_bootstrap'), load=True)
qrange_change_ds = storage.open_dataset(tas_result_path('qrange_change'), load=True)
tails_ds = storage.open_dataset(tas_result_path('tails'), load=True)

In [ ]:
#(t): The four changes compared with the sea ice
tas_change_ds = xr.Dataset({
    'mean': mean_bootstrap_ds['mean_change'],
    'width': qrange_change_ds['qrange_change'],
    'lower tail': tails_ds['lower_tail_change'],
    'upper tail': tails_ds['upper_tail_change'],
}).sel(experiment=EXPERIMENTS)
tas_change_ds

In [ ]:
#(t): Check: the cells, models and seasons the temperature and the sea ice share
dict(xr.align(tas_change_ds, ice_climatology_da, join='inner')[0].sizes)

### 5.1 The change in sea ice

Each experiment's ensemble-mean concentration over the final `WINDOW` years minus hist-nat's, with hist-nat's edge (dashed) and the experiment's (solid). Blue: more ice; red: less. Not tested: the edges and the temperature changes are what matter here.

In [ ]:
#(t): The change in concentration (%), every model, experiment and season (reindex: empty where no model has an experiment)
ice_change_da = ice_climatology_da.reindex(experiment=EXPERIMENTS) - ice_climatology_da.sel(experiment='hist-nat')
ice_change_da

#### What goes in

In [ ]:
#(t): MODEL's hist-nat and EXPERIMENT, and the change
model_ice_da = ice_climatology_da.sel(model=MODEL, season=SEASONS)
model_edges_da = sip.edge_pair(model_ice_da, EXPERIMENT)

In [ ]:
panels = fr.row_grid(
    [fr.Row(model_ice_da.sel(experiment='hist-nat'), concentration_scale, f'hist-nat\nfinal {WINDOW} years',
            edges=model_edges_da.sel(edge=['hist-nat'])),
     fr.Row(model_ice_da.sel(experiment=EXPERIMENT), concentration_scale, f'{EXPERIMENT}\nfinal {WINDOW} years',
            edges=model_edges_da.sel(edge=['experiment'])),
     fr.Row(ice_change_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS), concentration_change_scale,
            f'Change\n{EXPERIMENT} − hist-nat', edges=model_edges_da)],
    title=f'What goes into the change in sea ice: {MODEL}, {EXPERIMENT}',
)

#### One model

In [ ]:
panels = fr.map_grid(ice_change_da.sel(model=MODEL, season=SEASONS), concentration_change_scale,
                     row_dim='experiment', col_dim='season', edges=sip.edge_pair(model_ice_da, EXPERIMENTS),
                     title=f'{MODEL}: change in sea-ice concentration, final {WINDOW} years against hist-nat')

#### Every model

One figure per experiment.

In [ ]:
for experiment in EXPERIMENTS:
    panels = fr.map_grid(ice_change_da.sel(experiment=experiment, season=SEASONS), concentration_change_scale,
                         row_dim='model', col_dim='season',
                         edges=sip.edge_pair(ice_climatology_da.sel(season=SEASONS), experiment),
                         title=f'Change in sea-ice concentration, final {WINDOW} years against hist-nat, '
                               f'every model: {experiment}')

#### The models together

The multi-model median of the change, with the edges of the median concentration.

In [ ]:
median_ice_da = ice_climatology_da.sel(season=SEASONS).median('model')

In [ ]:
panels = fr.map_grid(ice_change_da.sel(season=SEASONS).median('model'), concentration_change_scale,
                     row_dim='experiment', col_dim='season', edges=sip.edge_pair(median_ice_da, EXPERIMENTS),
                     title=f'Change in sea-ice concentration, final {WINDOW} years against hist-nat, multi-model median')

#### As numbers

The change in extent (million km²) and in the edge's equivalent latitude (degrees; negative: further south).

In [ ]:
#(t): Each model's ensemble-mean extent over the final WINDOW years
ice_final_ds = sea_ice.final_mean(ice_extents_tree, WINDOW)
ice_final_ds

In [ ]:
#(t): The change in extent (million km²)
extent_change_da = (ice_final_ds['extent'].reindex(experiment=EXPERIMENTS)
                    - ice_final_ds['extent'].sel(experiment='hist-nat')) / 1e6
extent_change_da.sel(season=SEASONS).to_dataframe('Δ extent').unstack('season')['Δ extent'].round(2)

In [ ]:
#(t): The change in the edge's latitude (degrees)
edge_change_da = ice_edges_da.reindex(experiment=EXPERIMENTS) - ice_edges_da.sel(experiment='hist-nat')
edge_change_da.sel(season=SEASONS).to_dataframe('Δ edge').unstack('season')['Δ edge'].round(1)

### 5.2 Where the ice went: the temperature changes in each zone

Each ocean cell is put in a zone by its concentration in hist-nat's final `WINDOW` years and the experiment's (`sea_ice.ice_zones`):

| Zone | hist-nat | experiment |
|---|---|---|
| open water | < 15% | < 15% |
| ice in both | ≥ 15% | ≥ 15% |
| ice lost | ≥ 15% | < 15% |
| ice gained | < 15% | ≥ 15% |
| continent | land south of 60°S | |

Then each temperature change of notebook 03 is averaged over each zone (weighted by area). If the expectation holds, "ice lost" warms more than "open water" and "ice in both" in autumn and winter, and narrows (a negative width change, from a warming lower tail).

In [ ]:
#(t): Each cell's zone (its number in sea_ice.ZONES), every model, experiment and season
ice_zones_da = sea_ice.ice_zones(ice_climatology_da.sel(experiment='hist-nat'), ice_climatology_da.reindex(experiment=EXPERIMENTS))
ice_zones_da

In [ ]:
#(t): The area of each zone (million km²), the multi-model median
ice_zone_areas_da = sea_ice.zone_areas(ice_zones_da)
ice_zone_areas_da.sel(season=SEASONS).median('model').to_dataframe('area').unstack('zone')['area'].round(2)

#### What goes in

`MODEL` and `EXPERIMENT`: the zones, and the mean and width changes behind them, with both edges.

In [ ]:
model_change_ds = tas_change_ds.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS)
model_change_ds

In [ ]:
panels = fr.row_grid(
    [fr.Row(ice_zones_da.sel(model=MODEL, experiment=EXPERIMENT, season=SEASONS), zone_scale, 'Zone',
            edges=model_edges_da),
     fr.Row(model_change_ds['mean'], tas_scales.change, f'Δ mean\n{EXPERIMENT} − hist-nat', edges=model_edges_da),
     fr.Row(model_change_ds['width'], tas_scales.spread_change, f'Δ(Q95 − Q05)\n{EXPERIMENT} − hist-nat',
            edges=model_edges_da)],
    title=f'Where the ice changed, and the temperature: {MODEL}, {EXPERIMENT}, final {WINDOW} years',
)

#### Every model

In [ ]:
#(t): Each change averaged over each zone, every model, experiment and season
tas_zone_means_ds = sea_ice.zone_means(tas_change_ds, ice_zones_da)
tas_zone_means_ds

In [ ]:
fig, axes = sip.zone_changes(tas_zone_means_ds['mean'], f'Δ mean ({TAS_UNITS})', experiments=EXPERIMENTS, seasons=SEASONS,
                             title=f'Change in the mean temperature in each zone, final {WINDOW} years against hist-nat')

In [ ]:
fig, axes = sip.zone_changes(tas_zone_means_ds['width'], f'Δ(Q95 − Q05) ({TAS_UNITS})', experiments=EXPERIMENTS,
                             seasons=SEASONS,
                             title=f'Change in the width in each zone, final {WINDOW} years against hist-nat')

In [ ]:
fig, axes = sip.zone_changes(tas_zone_means_ds['lower tail'], f'Δ(Q50 − Q05) ({TAS_UNITS})', experiments=EXPERIMENTS,
                             seasons=SEASONS,
                             title=f'Change in the lower (cold) tail in each zone: negative where the cold extremes '
                                   f'warm more than the median')

In [ ]:
fig, axes = sip.zone_changes(tas_zone_means_ds['upper tail'], f'Δ(Q95 − Q50) ({TAS_UNITS})', experiments=EXPERIMENTS,
                             seasons=SEASONS,
                             title='Change in the upper (warm) tail in each zone')

In [ ]:
#(t): The multi-model medians as numbers: every experiment and change, the seasons and zones across
tas_zone_means_ds.sel(season=SEASONS).median('model').to_dataframe().unstack(['season', 'zone']).round(2).T

#### Save

In [ ]:
storage.save(ice_change_da.rename(SEA_ICE), result_path('ice_change'))

In [ ]:
storage.save(tas_zone_means_ds.assign(zone_area=ice_zone_areas_da), result_path('zone_means'))

### 5.3 Across models and forcings

One point per model and forcing: the change in extent against the change in the mean, and in the width, averaged over the cells with ice in hist-nat (where ice can be lost). If losing ice is what drives them, more loss (further left) goes with more warming and more narrowing; r is the correlation over all the points.

In [ ]:
#(t): The cells with ice (at least THRESHOLD %) in hist-nat's final years
hist_nat_ice_da = ice_climatology_da.sel(experiment='hist-nat') >= THRESHOLD

In [ ]:
#(t): Each temperature change averaged over them (cos-latitude weighted)
ice_region_change_ds = sig.area_mean(tas_change_ds.where(hist_nat_ice_da))
ice_region_change_ds

In [ ]:
panels = paper_panels.scatter_by_season(extent_change_da, ice_region_change_ds['mean'], 'Δ extent (million km²)',
                                        f'Δ mean over hist-nat\'s ice ({TAS_UNITS})', experiments=EXPERIMENTS,
                                        columns=SEASONS, style=fr.NOTEBOOK)

In [ ]:
panels = paper_panels.scatter_by_season(extent_change_da, ice_region_change_ds['width'], 'Δ extent (million km²)',
                                        f'Δ(Q95 − Q05) over hist-nat\'s ice ({TAS_UNITS})', experiments=EXPERIMENTS,
                                        columns=SEASONS, style=fr.NOTEBOOK)

### 5.4 Year to year: where the sea ice and the temperature move together (JASMIN)

In hist-nat, the correlation over every member and year between a cell's seasonal concentration and its seasonal temperature: strongly negative where a season with more ice is a colder season. That is where the ice sets the year-to-year variability of the temperature, so where losing it should change the width most. ERA5's (1979–2013, detrended) is in the last row, to see whether the models have the link in the right places. Only members in both the sea-ice and the temperature files count.

In [ ]:
#(t): The seasonal temperature notebook 01 saved
tas_season_tree = xr.open_datatree(paths.find(paths.seasonal('lesfmip', VARIABLE)), engine='zarr', chunks={})

In [ ]:
#(t): hist-nat's sea ice and temperature, every model, members along one dim (NaN where a model has fewer)
ice_hist_nat_da = tree_to_dataset(ice_season_tree.match('*/hist-nat'), ['model', 'experiment'])[SEA_ICE].sel(experiment='hist-nat')
tas_hist_nat_da = tree_to_dataset(tas_season_tree.match('*/hist-nat'), ['model', 'experiment'])[VARIABLE].sel(experiment='hist-nat')

In [ ]:
#(t): How many members each model has in both
shared_members_da = (ice_hist_nat_da.isel(year=0, season=0).notnull().any(['lat', 'lon'])
                     & tas_hist_nat_da.isel(year=0, season=0).notnull().any(['lat', 'lon'])).sum('member')
shared_members_da.to_series()

In [ ]:
%%time
#(t): The correlation over every member and year
coupling_da = xr.corr(ice_hist_nat_da, tas_hist_nat_da, dim=['member', 'year']).compute()
coupling_da

In [ ]:
#(t): ERA5's, from its detrended seasons over ERA5_YEARS
era5_tas_da = loading.open_era5_seasonal(VARIABLE).sel(year=ERA5_YEARS)
era5_coupling_da = xr.corr(ev.detrend(era5_ice_season_da.sel(year=ERA5_YEARS)), ev.detrend(era5_tas_da), dim='year')
era5_coupling_da

In [ ]:
#(t): The models and ERA5 as rows, and each one's own edge (hist-nat's final years; ERA5's 1979-2013)
coupling_rows_da = xr.concat([coupling_da, era5_coupling_da.expand_dims(model=['ERA5'])], dim='model')
coupling_edges_da = xr.concat([ice_climatology_da.sel(experiment='hist-nat', drop=True),
                               era5_climatology_da.expand_dims(model=['ERA5'])], dim='model').expand_dims(edge=['edge'])

In [ ]:
panels = fr.map_grid(coupling_rows_da.sel(season=SEASONS), correlation_scale, row_dim='model', col_dim='season',
                     edges=coupling_edges_da.sel(season=SEASONS),
                     title='Year to year, how the sea ice and the temperature move together: hist-nat, and ERA5')

In [ ]:
storage.save(coupling_rows_da.rename('correlation'), result_path('ice_tas_correlation'))

## What was saved

Under `results/siconc/<RUN>/`, in `paths.SCRATCH` (move them to `paths.DATA_DIR` by hand, as in notebook 01):

| Result | What | Read by |
|---|---|---|
| `extent` | each member's seasonal extent (km²), every model and experiment, and its continent | notebook 04 (7.6, the zonal edges), here |
| `climatology` | the ensemble-mean concentration over each experiment's final years (%) | notebook 04 (every map's edges), 07 (B1, D1–D4) |
| `era5_extent`, `era5_climatology`, `historical_climatology` | ERA5's seasonal extent; ERA5's and each model's historical concentration over 1979–2013 | |
| `era5_extent_test` | ERA5 among each model's members: the extent's mean, variability and trend | |
| `ice_change`, `zone_means` | the change in concentration; the temperature changes in each zone, and the zones' areas | |
| `ice_tas_correlation` | the year-to-year correlation of sea ice and temperature, hist-nat and ERA5 | |

The monthly sea-ice stores (full data) sit with the temperature's in `paths.LESFMIP_MONTHLY`, and ERA5's in `monthly/era5_siconc.zarr`.